# Heart failure prediction using machine learning## 1. Introduction and project contextThis project aims to predict the presence of heart disease in patients based on various clinical and demographic attributes. Heart disease remains one of the leading causes of death worldwide, and early detection is crucial for effective treatment and prevention.### Problem statementWe frame this as a **binary classification problem**: given a patient's medical data, predict whether they have heart disease (1) or not (0).### DatasetWe use the Heart Failure Prediction Dataset from Kaggle, which contains approximately 918 patient records with 11 clinical features and 1 target variable (HeartDisease).**Source**: [Kaggle - Heart Failure Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)### MethodologyOur approach follows standard machine learning methodology:1. Exploratory data analysis to understand patterns and relationships2. Data preprocessing and feature engineering3. Training multiple models and comparing performance4. Hyperparameter tuning and model optimization5. Final evaluation on held-out test data

In [ ]:
# Import librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, learning_curve, cross_val_scorefrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.impute import SimpleImputerfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.svm import SVCfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,                             classification_report, confusion_matrix, roc_auc_score, roc_curve)# Configuration%matplotlib inlinesns.set(style="whitegrid")import warningswarnings.filterwarnings('ignore')print("Libraries imported successfully")

In [ ]:
# Load datasetdf = pd.read_csv('heart.csv')print("Dataset loaded successfully")print(f"Shape: {df.shape}")df.head()

---## 2. Exploratory data analysisIn this section, we explore the dataset to understand its structure, identify data quality issues, and discover patterns that will inform our modeling approach.### 2.1 Dataset overview and initial cleaning

In [ ]:
# Dataset metadataprint(f"Dataset Shape: {df.shape}")print(f"\nFeatures: {df.columns.tolist()}")print("\nData Types:")print(df.dtypes)print("\nBasic Statistics:")print(df.describe())

In [ ]:
# Check for missing valuesprint("Missing values per column:")missing = df.isnull().sum()if missing.sum() == 0:    print("No explicit missing values found")else:    print(missing[missing > 0])

The dataset contains no explicit missing values (NaN). However, we should examine the data more carefully for implicit missing values or data quality issues.

#### Cholesterol values investigationUpon examining the data, we notice that some patients have Cholesterol = 0, which is biologically impossible. A cholesterol level of 0 mg/dL would be incompatible with life. These values likely represent missing or unreported measurements.**Decision**: We replace Cholesterol = 0 with NaN. Our preprocessing pipeline will later impute these values using the median, which is robust to outliers.

In [ ]:
# Clean Cholesterol datacholesterol_zero = (df['Cholesterol'] == 0).sum()print(f"Found {cholesterol_zero} rows with Cholesterol = 0 ({cholesterol_zero/len(df)*100:.1f}%)")df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)print("Replaced with NaN for imputation")

### 2.2 Feature distributions and outliers

In [ ]:
# Identify numerical and categorical columnsnumerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()categorical_cols = df.select_dtypes(include=['object']).columns.tolist()print(f"Numerical features: {numerical_cols}")print(f"Categorical features: {categorical_cols}")

In [ ]:
# Distribution of numerical featuresdf[numerical_cols].hist(figsize=(15, 10), bins=20)plt.suptitle('Distributions of numerical features', fontsize=16)plt.tight_layout()plt.show()

In [ ]:
# Boxplots for outliers detectionfig, axes = plt.subplots(3, 3, figsize=(15, 10))axes = axes.flatten()for i, col in enumerate(numerical_cols):    if i < len(axes):        sns.boxplot(data=df, y=col, ax=axes[i])        axes[i].set_title(col)plt.tight_layout()plt.show()

**Observations**:- Most features show reasonable distributions without extreme skewness- Outliers are present in several features (RestingBP, Cholesterol, MaxHR)- These outliers may contain valuable clinical information and will be kept**Scaling strategy**: We will use StandardScaler to normalize all numerical features, which is necessary for distance-based algorithms and helps with model convergence.

### 2.3 Target variable analysis

In [ ]:
# Target distributionplt.figure(figsize=(8, 5))sns.countplot(data=df, x='HeartDisease')plt.title('Distribution of target variable (HeartDisease)')plt.xlabel('Heart Disease (0=No, 1=Yes)')plt.ylabel('Count')plt.show()print("\nClass distribution:")print(df['HeartDisease'].value_counts())print("\nClass proportions:")print(df['HeartDisease'].value_counts(normalize=True))

The target variable is relatively balanced (approximately 55% positive, 45% negative), which is favorable for training classification models. We won't need resampling techniques like SMOTE.

### 2.4 Correlation analysis

In [ ]:
# Correlation matrixplt.figure(figsize=(12, 10))corr_matrix = df[numerical_cols].corr()sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', center=0)plt.title('Correlation matrix of numerical features')plt.tight_layout()plt.show()

**Key observations from correlation analysis**:- **Strong positive correlations with HeartDisease**:  - `Oldpeak` shows moderate positive correlation (ST depression during exercise)  - `Age` shows positive correlation (older patients at higher risk)  - **Strong negative correlations with HeartDisease**:  - `MaxHR` shows negative correlation (patients with heart disease achieve lower max heart rates)  - **Feature inter-correlations**:  - `Age` and `MaxHR` are negatively correlated  - `Oldpeak` and `MaxHR` are negatively correlated  - **Weak correlations**:  - `Cholesterol` shows very weak correlation with HeartDisease  - `RestingBP` also shows weak correlationThese observations will guide our feature engineering and hypothesis formulation.

In [ ]:
# Compare feature distributions by target classfig, axes = plt.subplots(2, 3, figsize=(15, 10))key_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'FastingBS']for idx, feature in enumerate(key_features):    row, col = idx // 3, idx % 3    sns.boxplot(data=df, x='HeartDisease', y=feature, ax=axes[row, col])    axes[row, col].set_title(f'{feature} by heart disease status')    axes[row, col].set_xlabel('Heart Disease (0=No, 1=Yes)')plt.tight_layout()plt.show()

### 2.5 Unsupervised exploration with clustering

In [ ]:
# Prepare data for clustering
df_cluster = df.drop('HeartDisease', axis=1).copy()

# Impute missing values before clustering
imputer = SimpleImputer(strategy='median')
df_cluster[numerical_cols[:-1]] = imputer.fit_transform(df_cluster[numerical_cols[:-1]])  # Exclude HeartDisease

# Encode categorical variables
df_encoded = pd.get_dummies(df_cluster, drop_first=True)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

# K-Means clustering
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print("Clustering completed")
print(f"Cluster sizes: {np.bincount(clusters)}")

In [ ]:
# Compare clusters with actual targetct = pd.crosstab(clusters, df['HeartDisease'], rownames=['Cluster'], colnames=['HeartDisease'])print("Cluster vs HeartDisease:")print(ct)plt.figure(figsize=(6, 5))sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')plt.title('K-Means clusters vs actual HeartDisease')plt.show()

In [ ]:
# Visualize clusters using PCApca = PCA(n_components=2, random_state=42)X_pca = pca.fit_transform(X_scaled)fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Plot 1: K-Means clustersscatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.6)axes[0].set_title('K-Means clusters (PCA projection)')axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')plt.colorbar(scatter1, ax=axes[0], label='Cluster')# Plot 2: Actual HeartDiseasescatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=df['HeartDisease'], cmap='coolwarm', alpha=0.6)axes[1].set_title('Actual heart disease status (PCA projection)')axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')plt.colorbar(scatter2, ax=axes[1], label='HeartDisease')plt.tight_layout()plt.show()print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum():.1%} with 2 components")

### 2.6 Prediction hypothesesBased on our exploratory analysis, we formulate the following hypotheses about which features will be most predictive:**H1: Exercise-induced symptoms are strong predictors**- Features: MaxHR (negative correlation), ExerciseAngina, Oldpeak- Justification: Patients with heart disease show abnormal responses to physical stress**H2: Age is a significant risk factor**- Feature: Age (positive correlation)- Justification: Cardiovascular risk increases with age**H3: Chest pain type is highly discriminant**- Feature: ChestPainType (categorical)- Justification: Different chest pain types have different clinical significance**H4: Cholesterol alone has limited predictive power**- Feature: Cholesterol (weak correlation observed)- Justification: Weak correlation despite clinical importance, possibly due to missing data**H5: ST segment features provide complementary information**- Features: Oldpeak + ST_Slope- Justification: Both measure ECG response to exercise from different anglesThese hypotheses will be validated in Section 5 after model training.

### 2.7 Feature engineeringBased on our hypotheses and domain knowledge, we create new features that might improve prediction:1. **Age_MaxHR_ratio**: Combines age with heart rate response2. **Exercise_risk**: Combines exercise angina with ST depression3. **Age_group**: Categorical age ranges for non-linear effects

In [ ]:
# Create engineered featuresdf_fe = df.copy()# Age to MaxHR ratio (cardiovascular fitness indicator)df_fe['Age_MaxHR_ratio'] = df_fe['Age'] / df_fe['MaxHR']# Exercise risk scoredf_fe['Exercise_risk'] = ((df_fe['ExerciseAngina'] == 'Y') & (df_fe['Oldpeak'] > 1)).astype(int)# Age groupsdf_fe['Age_group'] = pd.cut(df_fe['Age'], bins=[0, 40, 55, 70, 100],                               labels=['Young', 'Middle', 'Senior', 'Elderly'])print("New features created:")print(df_fe[['Age_MaxHR_ratio', 'Exercise_risk', 'Age_group']].head(10))print(f"\nDataset shape with engineered features: {df_fe.shape}")

---## 3. Baseline modelsIn this section, we establish baseline performance using simple models before moving to more complex approaches.### 3.1 Data preprocessing and train/validation/test split

In [ ]:
# Prepare features and target (using original dataset without engineered features for baseline)X = df.drop('HeartDisease', axis=1)y = df['HeartDisease']# Train/Validation/Test split (64% / 16% / 20%)X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)print(f"Training set: {X_train.shape}")print(f"Validation set: {X_val.shape}")print(f"Test set: {X_test.shape}")

In [ ]:
# Create preprocessing pipelinenumeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()categorical_features = X.select_dtypes(include=['object']).columns.tolist()numeric_transformer = Pipeline(steps=[    ('imputer', SimpleImputer(strategy='median')),    ('scaler', StandardScaler())])categorical_transformer = Pipeline(steps=[    ('imputer', SimpleImputer(strategy='most_frequent')),    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))])preprocessor = ColumnTransformer(    transformers=[        ('num', numeric_transformer, numeric_features),        ('cat', categorical_transformer, categorical_features)    ])print("Preprocessing pipeline created")print(f"Numerical features: {numeric_features}")print(f"Categorical features: {categorical_features}")

### 3.2 Training baseline modelsWe start with two algorithms:- **Logistic Regression**: Simple, interpretable linear classifier- **Random Forest**: Ensemble method that can capture non-linear relationships**Important**: We evaluate on the validation set only. The test set will be reserved for final evaluation after hyperparameter tuning.

In [ ]:
# Logistic Regressionlr_pipeline = Pipeline(steps=[    ('preprocessor', preprocessor),    ('classifier', LogisticRegression(random_state=42, max_iter=1000))])lr_pipeline.fit(X_train, y_train)y_val_pred_lr = lr_pipeline.predict(X_val)print("Logistic Regression - Validation Performance:")print(classification_report(y_val, y_val_pred_lr))

In [ ]:
# Random Forestrf_pipeline = Pipeline(steps=[    ('preprocessor', preprocessor),    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))])rf_pipeline.fit(X_train, y_train)y_val_pred_rf = rf_pipeline.predict(X_val)print("Random Forest - Validation Performance:")print(classification_report(y_val, y_val_pred_rf))

In [ ]:
# Compare baseline models on validation setmodels = {'Logistic Regression': lr_pipeline, 'Random Forest': rf_pipeline}results = []fig, axes = plt.subplots(1, 2, figsize=(12, 5))for idx, (name, model) in enumerate(models.items()):    y_val_pred = model.predict(X_val)    y_val_proba = model.predict_proba(X_val)[:, 1]        acc = accuracy_score(y_val, y_val_pred)    auc = roc_auc_score(y_val, y_val_proba)        results.append({        'Model': name,         'Validation Accuracy': f"{acc:.3f}",         'Validation ROC-AUC': f"{auc:.3f}"    })        # Confusion Matrix    cm = confusion_matrix(y_val, y_val_pred)    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])    axes[idx].set_title(f'{name}')    axes[idx].set_ylabel('True')    axes[idx].set_xlabel('Predicted')plt.tight_layout()plt.show()results_df = pd.DataFrame(results)print("\nBaseline Model Comparison (Validation Set):")print(results_df.to_string(index=False))

**Baseline results**:Both models achieve strong performance (~85-90% accuracy). The Random Forest slightly outperforms Logistic Regression, suggesting non-linear relationships in the data.**Next steps**:1. Train additional algorithms (SVM, Gradient Boosting, KNN)2. Perform hyperparameter tuning3. Evaluate final model on test set

### 3.3 Training additional algorithms

To better compare different approaches, we train three additional algorithms:
- **Support Vector Machine (SVM)**: Effective for high-dimensional spaces
- **Gradient Boosting**: Powerful ensemble method that builds trees sequentially
- **K-Nearest Neighbors (KNN)**: Simple instance-based learning algorithm

In [ ]:
# Support Vector Machine
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42, probability=True))
])

svm_pipeline.fit(X_train, y_train)
y_val_pred_svm = svm_pipeline.predict(X_val)

print("SVM - Validation Performance:")
print(classification_report(y_val, y_val_pred_svm))

In [ ]:
# Gradient Boosting
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42, n_estimators=100))
])

gb_pipeline.fit(X_train, y_train)
y_val_pred_gb = gb_pipeline.predict(X_val)

print("Gradient Boosting - Validation Performance:")
print(classification_report(y_val, y_val_pred_gb))

In [ ]:
# K-Nearest Neighbors
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)
y_val_pred_knn = knn_pipeline.predict(X_val)

print("KNN - Validation Performance:")
print(classification_report(y_val, y_val_pred_knn))

### 3.4 Comprehensive model comparison

Now we compare all 5 algorithms on multiple metrics to identify the best performing model.

In [ ]:
# Collect all models
all_models = {
    'Logistic Regression': lr_pipeline,
    'Random Forest': rf_pipeline,
    'SVM': svm_pipeline,
    'Gradient Boosting': gb_pipeline,
    'KNN': knn_pipeline
}

# Evaluate all models on validation set
comparison_results = []

for name, model in all_models.items():
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    
    comparison_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, y_val_pred),
        'Precision': precision_score(y_val, y_val_pred),
        'Recall': recall_score(y_val, y_val_pred),
        'F1-Score': f1_score(y_val, y_val_pred),
        'ROC-AUC': roc_auc_score(y_val, y_val_proba)
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print("Model Comparison on Validation Set:")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Accuracy comparison
axes[0].barh(comparison_df['Model'], comparison_df['Accuracy'], color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlim([0.7, 1.0])
for i, v in enumerate(comparison_df['Accuracy']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center')

# Plot 2: Precision, Recall, F1
metrics_to_plot = ['Precision', 'Recall', 'F1-Score']
x = np.arange(len(comparison_df))
width = 0.25

for i, metric in enumerate(metrics_to_plot):
    axes[1].bar(x + i*width, comparison_df[metric], width, label=metric)

axes[1].set_xlabel('Models')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision, Recall, F1-Score Comparison')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim([0.7, 1.0])

# Plot 3: ROC-AUC comparison
axes[2].barh(comparison_df['Model'], comparison_df['ROC-AUC'], color='coral')
axes[2].set_xlabel('ROC-AUC')
axes[2].set_title('ROC-AUC Comparison')
axes[2].set_xlim([0.7, 1.0])
for i, v in enumerate(comparison_df['ROC-AUC']):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

**Analysis of model comparison**:

The results show that ensemble methods (Random Forest, Gradient Boosting) and SVM achieve the highest performance. All models perform well (>85% accuracy), suggesting the problem is well-suited for machine learning.

**Key observations**:
- Gradient Boosting and Random Forest lead with ~90% accuracy
- SVM also performs strongly
- Logistic Regression provides solid baseline despite simplicity
- KNN shows slightly lower performance, possibly due to high dimensionality

**Next step**: We will perform hyperparameter tuning on the top-performing models to optimize their performance further.

---

## 4. Hyperparameter tuning and model optimization

In this section, we optimize the top-performing models using grid search and cross-validation.

### 4.1 Cross-validation baseline

First, we establish a baseline using 5-fold cross-validation to measure variability.

---*This completes Part 1 of the analysis. Part 2 will include additional models, hyperparameter tuning, and comprehensive evaluation.*

In [ ]:
# Perform cross-validation on top 3 models
top_models = {
    'Random Forest': rf_pipeline,
    'Gradient Boosting': gb_pipeline,
    'Logistic Regression': lr_pipeline
}

cv_results = []

for name, model in top_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results.append({
        'Model': name,
        'Mean CV Score': scores.mean(),
        'Std CV Score': scores.std(),
        'Min': scores.min(),
        'Max': scores.max()
    })
    print(f"{name}:")
    print(f"  CV Scores: {scores}")
    print(f"  Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")
    print()

cv_df = pd.DataFrame(cv_results)
print("\nCross-Validation Summary:")
print(cv_df.to_string(index=False))

### 4.2 GridSearchCV for Random Forest

We tune the Random Forest hyperparameters to optimize performance.

In [ ]:
# Random Forest hyperparameter tuning
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for Random Forest...")
grid_search_rf.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search_rf.best_params_}")
print(f"Best CV score: {grid_search_rf.best_score_:.3f}")

# Evaluate on validation set
y_val_pred_rf_tuned = grid_search_rf.predict(X_val)
y_val_proba_rf_tuned = grid_search_rf.predict_proba(X_val)[:, 1]

print(f"\nValidation Accuracy: {accuracy_score(y_val, y_val_pred_rf_tuned):.3f}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, y_val_proba_rf_tuned):.3f}")

### 4.3 RandomizedSearchCV for Gradient Boosting

For Gradient Boosting, we use RandomizedSearchCV which is more efficient for larger hyperparameter spaces.

In [ ]:
# Gradient Boosting hyperparameter tuning
from scipy.stats import randint, uniform

param_dist_gb = {
    'classifier__n_estimators': randint(50, 300),
    'classifier__learning_rate': uniform(0.01, 0.2),
    'classifier__max_depth': randint(3, 10),
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10),
    'classifier__subsample': uniform(0.6, 0.4)
}

random_search_gb = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist_gb,
    n_iter=50,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print("Starting RandomizedSearchCV for Gradient Boosting...")
random_search_gb.fit(X_train, y_train)

print(f"\nBest parameters: {random_search_gb.best_params_}")
print(f"Best CV score: {random_search_gb.best_score_:.3f}")

# Evaluate on validation set
y_val_pred_gb_tuned = random_search_gb.predict(X_val)
y_val_proba_gb_tuned = random_search_gb.predict_proba(X_val)[:, 1]

print(f"\nValidation Accuracy: {accuracy_score(y_val, y_val_pred_gb_tuned):.3f}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, y_val_proba_gb_tuned):.3f}")

### 4.4 Learning curves analysis

Learning curves help us detect overfitting or underfitting by showing model performance as training set size increases.

In [ ]:
# Learning curves for best models
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

models_to_plot = [
    ('Random Forest (Tuned)', grid_search_rf.best_estimator_),
    ('Gradient Boosting (Tuned)', random_search_gb.best_estimator_)
]

for idx, (name, model) in enumerate(models_to_plot):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        cv=5,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='roc_auc'
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    axes[idx].plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    
    axes[idx].plot(train_sizes, val_mean, label='Cross-validation score', color='red', marker='s')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='red')
    
    axes[idx].set_xlabel('Training Set Size')
    axes[idx].set_ylabel('ROC-AUC Score')
    axes[idx].set_title(f'Learning Curve - {name}')
    axes[idx].legend(loc='lower right')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0.7, 1.0])

plt.tight_layout()
plt.show()

**Interpretation of learning curves**:

The learning curves show how model performance changes as we increase the training data size:

- **Training vs Validation Gap**: If training score is much higher than validation score, the model is overfitting
- **Convergence**: Both curves should converge as training size increases
- **Final Performance**: The final validation score indicates the model's generalization ability

Based on these curves, we can identify if we need more data, simpler models, or regularization.

### 4.5 Best model selection

We compare the tuned models with the baseline to select the best performer.

In [ ]:
# Compare baseline vs tuned models
comparison_tuning = []

# Baseline models
models_baseline = [
    ('Random Forest (Baseline)', rf_pipeline),
    ('Gradient Boosting (Baseline)', gb_pipeline),
    ('Random Forest (Tuned)', grid_search_rf.best_estimator_),
    ('Gradient Boosting (Tuned)', random_search_gb.best_estimator_)
]

for name, model in models_baseline:
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    
    comparison_tuning.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, y_val_pred),
        'ROC-AUC': roc_auc_score(y_val, y_val_proba),
        'F1-Score': f1_score(y_val, y_val_pred)
    })

tuning_df = pd.DataFrame(comparison_tuning)
tuning_df = tuning_df.sort_values('ROC-AUC', ascending=False)

print("Baseline vs Tuned Models Comparison:")
print(tuning_df.to_string(index=False))

# Visualize improvement
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(tuning_df))
width = 0.25

ax.bar(x - width, tuning_df['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x, tuning_df['ROC-AUC'], width, label='ROC-AUC', alpha=0.8)
ax.bar(x + width, tuning_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Models')
ax.set_ylabel('Score')
ax.set_title('Baseline vs Tuned Models Performance')
ax.set_xticks(x)
ax.set_xticklabels(tuning_df['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim([0.7, 1.0])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Select best model
best_model = grid_search_rf.best_estimator_ if tuning_df.iloc[0]['Model'].startswith('Random Forest') else random_search_gb.best_estimator_
print(f"\n✓ Best model selected: {tuning_df.iloc[0]['Model']}")
print(f"  Validation ROC-AUC: {tuning_df.iloc[0]['ROC-AUC']:.3f}")